<a href="https://colab.research.google.com/github/trsantanarural/predicao-falhas-tarefas/blob/main/notebooks/06_intervalos_bootstrap.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 06. Intervalos de confiança por bootstrap de jobs

**Entradas:** `saida/classificacao_predicoes_teste.parquet`, `saida/classificacao_limiares.csv` e `saida/classificacao_metricas.csv` (notebook 05)
**Saídas:** `saida/classificacao_bootstrap.csv` e `saida/classificacao_bootstrap_diferencas.csv`

As tarefas de um mesmo job são correlacionadas, então a reamostragem sorteia **jobs** (com reposição) e recalcula as métricas sobre as tarefas dos jobs sorteados, mantendo fixo o limiar escolhido na validação. Os limites do intervalo de 95% são os percentis 2,5 e 97,5 de 2.000 reamostragens. As diferenças entre variantes e entre algoritmos usam as mesmas reamostragens (diferença pareada). Um intervalo que inclui zero não é interpretado como diferença.

Não há novo ajuste de modelo: o notebook só lê as predições do teste. Roda em poucos minutos.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
BASE = '/content/drive/MyDrive/artigo_predicao'
SAIDA = f'{BASE}/saida'

import numpy as np, pandas as pd
B = 2000          # reamostragens de jobs
SEED = 0
DOM_TESTE = 6449919400   # job que concentra mais falhas no teste (notebook 04)


In [ ]:
# Predições do teste (notebook 05), limiares da validação e métricas já calculadas
d = pd.read_parquet(f'{SAIDA}/classificacao_predicoes_teste.parquet')
lim = pd.read_csv(f'{SAIDA}/classificacao_limiares.csv').set_index(['variante', 'algoritmo'])['limiar']
met = pd.read_csv(f'{SAIDA}/classificacao_metricas.csv')
modelos = [c for c in d.columns if ' | ' in c]
jobs, inv = np.unique(d.job_id.values, return_inverse=True)
nj = len(jobs)
y_all = d.falha.values.astype(float)
print(f'{len(d):,} tarefas de teste, {nj:,} jobs, {len(modelos)} modelos')

# Contagem de vezes que cada job entra em cada reamostragem (linha 0 = amostra original)
rng = np.random.RandomState(SEED)
cnt = np.stack([np.bincount(rng.randint(0, nj, nj), minlength=nj) for _ in range(B)]).astype(np.float32)
cnt = np.vstack([np.ones((1, nj), np.float32), cnt])


In [ ]:
def prep(score, mask):
    """Ordena por escore decrescente; marca o fim de cada grupo de escores empatados."""
    s = score[mask]; y = y_all[mask]; idx = np.where(mask)[0]
    o = np.argsort(-s, kind='stable'); s, y, j = s[o], y[o], inv[idx][o]
    return s, y, j, np.r_[s[1:] != s[:-1], True]

def curvas(y, j, last, C):
    """ROC-AUC (trapézios) e PR-AUC (precisão média) ponderadas pela contagem de cada job."""
    W = C[:, j]
    tp = np.cumsum(W * y, axis=1)[:, last]; fp = np.cumsum(W * (1 - y), axis=1)[:, last]
    P, N = tp[:, -1:], fp[:, -1:]
    tpr = np.hstack([np.zeros((len(C), 1)), tp / P]); fpr = np.hstack([np.zeros((len(C), 1)), fp / N])
    roc = np.sum(np.diff(fpr, axis=1) * (tpr[:, 1:] + tpr[:, :-1]) / 2, axis=1)
    ap = np.sum(np.diff(tpr, axis=1) * tp / np.maximum(tp + fp, 1e-12), axis=1)
    return roc, ap

def no_limiar(s, y, j, t, C):
    W = C[:, j]; pred = s >= t
    tp = (W * (y * pred)).sum(1); fp = (W * ((1 - y) * pred)).sum(1); fn = (W * (y * ~pred)).sum(1)
    p = tp / np.maximum(tp + fp, 1e-12); r = tp / np.maximum(tp + fn, 1e-12)
    return p, r, 2 * p * r / np.maximum(p + r, 1e-12)


In [ ]:
recortes = {'todos os jobs': np.ones(len(d), bool), 'sem o job dominante': d.job_id.values != DOM_TESTE}
res = {}
for rec, mask in recortes.items():
    for m in modelos:
        var, alg = m.split(' | ')
        s, y, j, last = prep(d[m].values.astype(np.float64), mask)
        v = {k: np.empty(B + 1) for k in ('roc', 'ap', 'p', 'r', 'f')}
        for a in range(0, B + 1, 100):
            C = cnt[a:a + 100]
            v['roc'][a:a + 100], v['ap'][a:a + 100] = curvas(y, j, last, C)
            v['p'][a:a + 100], v['r'][a:a + 100], v['f'][a:a + 100] = no_limiar(s, y, j, lim[(var, alg)], C)
        res[(rec, var, alg)] = v
    print(rec, 'ok')

# Conferência: a linha 0 (amostra original) deve reproduzir as métricas do notebook 05
rn = {'todos os jobs': 'todos os jobs', 'sem o job dominante': 'sem os jobs dominantes'}
dif = []
for (rec, var, alg), v in res.items():
    r = met[(met.variante == var) & (met.algoritmo == alg) & (met.conjunto == 'teste') & (met.recorte == rn[rec])].iloc[0]
    dif.append(max(abs(v['roc'][0] - r['ROC-AUC']), abs(v['ap'][0] - r['PR-AUC'])))
print('maior diferença em relação ao notebook 05 (ROC-AUC e PR-AUC):', round(max(dif), 4))


In [ ]:
NOMES = {'roc': 'ROC-AUC', 'ap': 'PR-AUC', 'p': 'precisão (falha)', 'r': 'recall (falha)', 'f': 'F1 (falha)'}
q = lambda x: (np.percentile(x[1:], 2.5), np.percentile(x[1:], 97.5))

lin = []
for (rec, var, alg), v in res.items():
    for k, n in NOMES.items():
        lo, hi = q(v[k]); lin.append(dict(recorte=rec, variante=var, algoritmo=alg, métrica=n, estimativa=v[k][0], ic95_inf=lo, ic95_sup=hi))
pd.DataFrame(lin).to_csv(f'{SAIDA}/classificacao_bootstrap.csv', index=False, float_format='%.6f')

# Diferenças pareadas (mesmas reamostragens): variante menos principal, no mesmo algoritmo, e XGBoost menos RF
VARS = ['com tempo observado', 'filtro de Pearson', 'tempo observado + filtro de Pearson', 'extensão com atributos de contexto']
lin = []
for rec in recortes:
    for alg in ('RF', 'XGBoost'):
        for var in VARS:
            for k in ('roc', 'ap', 'f'):
                x = res[(rec, var, alg)][k] - res[(rec, 'principal', alg)][k]; lo, hi = q(x)
                lin.append(dict(recorte=rec, comparação=f'{var} menos principal', algoritmo=alg, métrica=NOMES[k], diferença=x[0], ic95_inf=lo, ic95_sup=hi, ic_exclui_zero=bool(lo > 0 or hi < 0)))
    for k in ('roc', 'ap', 'f'):
        x = res[(rec, 'principal', 'XGBoost')][k] - res[(rec, 'principal', 'RF')][k]; lo, hi = q(x)
        lin.append(dict(recorte=rec, comparação='XGBoost menos RF (principal)', algoritmo='', métrica=NOMES[k], diferença=x[0], ic95_inf=lo, ic95_sup=hi, ic_exclui_zero=bool(lo > 0 or hi < 0)))
dif = pd.DataFrame(lin)
dif.to_csv(f'{SAIDA}/classificacao_bootstrap_diferencas.csv', index=False, float_format='%.6f')
print(f'{dif.ic_exclui_zero.sum()} de {len(dif)} diferenças têm intervalo que exclui zero')
dif[dif.ic_exclui_zero]
